In [1]:
import pandas as pd
import numpy as np
import joblib
import torch
import torch.nn as nn
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# 1. Load Data
df_features = pd.read_parquet('../data/processed/processed_features.parquet')
track_mapping = joblib.load('../saved_models/preprocessors/track_mapping.pkl')
trackid_to_index = joblib.load('../saved_models/preprocessors/trackid_to_index.pkl')

track_ids = df_features['track_id'].values
X = df_features.drop(columns=['track_id']).values 

# 2. Load Baseline & RBM Nearest Neighbors
baseline_model = joblib.load('../saved_models/baseline_nn_model.pkl')
rbm_nn_model = joblib.load('../saved_models/rbm_nn_model.pkl')

# 3. Load Markov Chain Arrays
markov_indices = np.load('../saved_models/markov_indices.npy')
markov_probs = np.load('../saved_models/markov_probs.npy')

# 4. Load RBM PyTorch Weights
class RBM(nn.Module):
    def __init__(self, n_visible, n_hidden):
        super(RBM, self).__init__()
        self.encoder = nn.Linear(n_visible, n_hidden)
        self.decoder = nn.Linear(n_hidden, n_visible)

    def forward(self, x):
        hidden = torch.sigmoid(self.encoder(x))
        reconstructed = torch.sigmoid(self.decoder(hidden))
        return reconstructed, hidden

n_visible = X.shape[1]
n_hidden = 64
rbm_model = RBM(n_visible, n_hidden)
rbm_model.load_state_dict(torch.load('../saved_models/rbm_weights.pth', weights_only=True))
rbm_model.eval()

# Pre-calculate RBM hidden space for the whole dataset
X_tensor = torch.FloatTensor(X)
with torch.no_grad():
    _, X_hidden = rbm_model(X_tensor)
X_hidden_np = X_hidden.numpy()

print("All models loaded successfully!")

All models loaded successfully!


In [2]:
def get_hybrid_recommendations(track_id, n_recommendations=5, w_rbm=0.4, w_markov=0.4, w_baseline=0.2):
    idx = trackid_to_index[track_id]
    
    # --- 1. Get Candidates from RBM (Latent Space) ---
    _, rbm_indices = rbm_nn_model.kneighbors(X_hidden_np[idx].reshape(1, -1))
    rbm_candidates = rbm_indices[0][1:21] # Top 20, skip self
    
    # --- 2. Get Candidates from Markov Chain (Sequential) ---
    step1_indices = markov_indices[idx]
    step1_probs = markov_probs[idx]
    
    markov_candidates_set = set()
    markov_scores_raw = {}
    
    # 2-Step Markov Logic
    for i in range(len(step1_indices)):
        b_idx = step1_indices[i]
        prob_a_to_b = step1_probs[i]
        
        step2_indices = markov_indices[b_idx]
        step2_probs = markov_probs[b_idx]
        
        for j in range(len(step2_indices)):
            c_idx = step2_indices[j]
            if c_idx == idx: continue
            
            combined_prob = prob_a_to_b * step2_probs[j]
            markov_candidates_set.add(c_idx)
            markov_scores_raw[c_idx] = markov_scores_raw.get(c_idx, 0) + combined_prob

    # Combine candidates from both models
    all_candidates = set(rbm_candidates.tolist()) | markov_candidates_set
    # Remove the input track itself
    all_candidates.discard(idx) 
    
    # --- 3. Score the Candidates using Baseline (Content-Based) ---
    # We only calculate cosine similarity for our small pool of candidates!
    baseline_scores_raw = {}
    input_vec = X[idx].reshape(1, -1)
    
    for c_idx in all_candidates:
        candidate_vec = X[c_idx].reshape(1, -1)
        # Cosine similarity = 1 - cosine distance
        dist = np.linalg.norm(input_vec - candidate_vec) # Simplified for speed, or use sklearn
        # Actually, let's use the pre-fitted baseline model for accuracy
        _, dists = baseline_model.kneighbors(candidate_vec)
        # Find distance from input to candidate
        _, input_dists = baseline_model.kneighbors(input_vec)
        # To keep it simple and fast: calculate direct cosine similarity
        sim = np.dot(input_vec[0], candidate_vec[0]) / (np.linalg.norm(input_vec[0]) * np.linalg.norm(candidate_vec[0]))
        baseline_scores_raw[c_idx] = sim

    # --- 4. Normalize Scores (Min-Max Scaling) ---
    # This ensures no model dominates just because its raw numbers are larger
    def normalize(scores_dict):
        if not scores_dict: return {}
        vals = list(scores_dict.values())
        min_val, max_val = min(vals), max(vals)
        if max_val == min_val: return {k: 0.5 for k in scores_dict}
        return {k: (v - min_val) / (max_val - min_val) for k, v in scores_dict.items()}

    norm_rbm = {idx: 1.0 for idx in rbm_candidates if idx in all_candidates} # RBM candidates get high base score
    norm_markov = normalize(markov_scores_raw)
    norm_baseline = normalize(baseline_scores_raw)

    # --- 5. Final Weighted Fusion ---
    final_scores = {}
    for c_idx in all_candidates:
        score_r = norm_rbm.get(c_idx, 0.0)
        score_m = norm_markov.get(c_idx, 0.0)
        score_b = norm_baseline.get(c_idx, 0.0)
        
        final_scores[c_idx] = (w_rbm * score_r) + (w_markov * score_m) + (w_baseline * score_b)

    # Sort and return
    sorted_tracks = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
    
    recommendations = []
    for c_idx, score in sorted_tracks[:n_recommendations]:
        rec_track_id = track_ids[c_idx]
        recommendations.append(track_mapping[rec_track_id])
        
    return recommendations

print("Hybrid function defined successfully!")

Hybrid function defined successfully!


In [3]:
import os
import json

os.makedirs('../saved_models', exist_ok=True)

hybrid_config = {
    "weights": {
        "rbm": 0.4,
        "markov": 0.4,
        "baseline": 0.2
    },
    "description": "Weighted ensemble of Latent Space (RBM), Sequential Flow (Markov), and Audio Features (Baseline)."
}

with open('../saved_models/hybrid_config.json', 'w') as f:
    json.dump(hybrid_config, f)

print("Hybrid configuration saved to '../saved_models/hybrid_config.json'")

Hybrid configuration saved to '../saved_models/hybrid_config.json'


In [4]:
# Pick a test track
test_track_id = track_ids[500] 
print(f"\n--- Testing HYBRID Model ---")
print(f"Input Track: {track_mapping[test_track_id]['track_name']} by {track_mapping[test_track_id]['artists']} ({track_mapping[test_track_id]['track_genre']})\n")

recs = get_hybrid_recommendations(test_track_id, n_recommendations=5)

print("Hybrid Recommendations:")
for i, rec in enumerate(recs, 1):
    print(f"{i}. {rec['track_name']} by {rec['artists']} [{rec['track_genre']}]")


--- Testing HYBRID Model ---
Input Track: A Thousand Years - Acoustic by Jonah Baker (acoustic)

Hybrid Recommendations:
1. Incomparável Amor - Acústico by Gabriel Brito [brazil]
2. North by Sleeping At Last [ambient]
3. You Were Supposed to Be Different by Aron Wright [acoustic]
4. To Whom It May Concern by The Civil Wars [acoustic]
5. Here, There, and Everywhere by Sweet Little Band [children]
